In [5]:
from IPython.display import HTML

HTML("""

<h2 class="sr-only">
Simulador cinemático do Otto
</h2>

<style>
.ow{font-family:monospace;padding:0.5rem 0;}
.ow-3col{display:grid;grid-template-columns:1fr 1fr 1fr;gap:8px;margin-bottom:8px;}
.ow-panel{background:#fafafa;border:1px solid #ddd;border-radius:8px;padding:7px;}
.ow-lbl{font-size:10px;letter-spacing:0.08em;text-transform:uppercase;color:#555;margin-bottom:4px;}
.ow-ctrl{background:#fff;border:1px solid #ddd;border-radius:8px;padding:11px;margin-bottom:8px;}
.ow-row{display:flex;align-items:center;gap:6px;margin-bottom:6px;}
.ow-row label{font-size:11px;min-width:120px;}
.ow-row input[type=range]{flex:1;}
.ow-row span{font-size:11px;min-width:45px;text-align:right;}
.ow-btns{display:flex;gap:6px;flex-wrap:wrap;margin-top:8px;}
.ow-btn{
padding:5px 10px;
font-size:11px;
border-radius:6px;
border:1px solid #bbb;
background:#f3f3f3;
cursor:pointer;
}
.ow-btn.on{
background:#185FA5;
color:white;
border-color:#185FA5;
}
</style>

<div class="ow">

<div class="ow-3col">

<div class="ow-panel">
<div class="ow-lbl">Vista superior XY</div>
<canvas id="cvT" width="220" height="220"></canvas>
</div>

<div class="ow-panel">
<div class="ow-lbl">Vista frontal XZ</div>
<canvas id="cvF" width="220" height="220"></canvas>
</div>

<div class="ow-panel">
<div class="ow-lbl">Trajetória global</div>
<canvas id="cvG" width="220" height="220"></canvas>
</div>

</div>

<div class="ow-ctrl">

<div class="ow-row">
<label>Amplitude pernas</label>
<input type="range" id="amp" min="5" max="45" value="30">
<span id="vamp">30°</span>
</div>

<div class="ow-row">
<label>Amplitude pés</label>
<input type="range" id="foot" min="0" max="35" value="18">
<span id="vfoot">18°</span>
</div>

<div class="ow-row">
<label>Período</label>
<input type="range" id="per" min="400" max="2500" value="1000">
<span id="vper">1000</span>
</div>

<div class="ow-btns">
<button class="ow-btn on" id="bWF" onclick="preset('wf')">Walk ↑</button>
<button class="ow-btn" id="bWB" onclick="preset('wb')">Walk ↓</button>
<button class="ow-btn" id="bTL" onclick="preset('tl')">Turn ←</button>
<button class="ow-btn" id="bTR" onclick="preset('tr')">Turn →</button>
<button class="ow-btn" id="playBtn" onclick="togglePlay()">▶</button>
</div>

</div>

</div>

<script>
(function(){

// =====================================================
// Canvas
// =====================================================

var cvT=document.getElementById("cvT");
var ctT=cvT.getContext("2d");

var cvF=document.getElementById("cvF");
var ctF=cvF.getContext("2d");

var cvG=document.getElementById("cvG");
var ctG=cvG.getContext("2d");

// =====================================================
// Estado
// =====================================================

var simT=0;
var playing=false;
var lastTS=null;

var globalX=0;
var globalY=0;
var heading=0;

var trail=[];

var mode="wf";

// =====================================================
// Parâmetros
// =====================================================

var LEG=80;
var HIP=26;
var BODYW=55;
var BODYH=40;

// =====================================================
// Presets
// =====================================================

window.preset=function(m){

mode=m;

document.querySelectorAll(".ow-btn").forEach(function(b){
b.classList.remove("on");
});

if(m==="wf")document.getElementById("bWF").classList.add("on");
if(m==="wb")document.getElementById("bWB").classList.add("on");
if(m==="tl")document.getElementById("bTL").classList.add("on");
if(m==="tr")document.getElementById("bTR").classList.add("on");

// NÃO reinicia trajetória

};

// =====================================================
// Play
// =====================================================

window.togglePlay=function(){

playing=!playing;

document.getElementById("playBtn").textContent=
playing?"⏹":"▶";

if(playing){
lastTS=null;
}

};

// =====================================================
// Sliders
// =====================================================

function pars(){

var A=+document.getElementById("amp").value;
var F=+document.getElementById("foot").value;
var T=+document.getElementById("per").value;

document.getElementById("vamp").textContent=A+"°";
document.getElementById("vfoot").textContent=F+"°";
document.getElementById("vper").textContent=T;

return{
A:A*Math.PI/180,
F:F*Math.PI/180,
T:T
};

}

// =====================================================
// Grade
// =====================================================

function grid(ctx,w,h){

ctx.strokeStyle="#eee";
ctx.lineWidth=1;

for(var i=0;i<w;i+=20){
ctx.beginPath();
ctx.moveTo(i,0);
ctx.lineTo(i,h);
ctx.stroke();
}

for(var j=0;j<h;j+=20){
ctx.beginPath();
ctx.moveTo(0,j);
ctx.lineTo(w,j);
ctx.stroke();
}

}

// =====================================================
// Vista superior
// =====================================================

function drawTop(thL,thR){

ctT.clearRect(0,0,220,220);

grid(ctT,220,220);

var cx=110;
var cy=110;

// quadris
var hx1=cx-HIP;
var hx2=cx+HIP;

// pés alinhados no mesmo centro do tronco
var fy=cy+55;

var fx1=hx1+LEG*0.35*Math.sin(thL);
var fx2=hx2+LEG*0.35*Math.sin(thR);

// corpo
ctT.fillStyle="#2c7be5";
ctT.fillRect(cx-BODYW/2,cy-BODYH/2,BODYW,BODYH);

// linha central
ctT.strokeStyle="rgba(0,0,0,0.2)";
ctT.beginPath();
ctT.moveTo(cx,20);
ctT.lineTo(cx,200);
ctT.stroke();

// pernas
ctT.lineWidth=4;

ctT.strokeStyle="#1D9E75";
ctT.beginPath();
ctT.moveTo(hx1,cy);
ctT.lineTo(fx1,fy);
ctT.stroke();

ctT.strokeStyle="#378ADD";
ctT.beginPath();
ctT.moveTo(hx2,cy);
ctT.lineTo(fx2,fy);
ctT.stroke();

// pés
ctT.fillStyle="#1D9E75";
ctT.fillRect(fx1-12,fy-6,24,12);

ctT.fillStyle="#378ADD";
ctT.fillRect(fx2-12,fy-6,24,12);

}

// =====================================================
// Vista frontal
// =====================================================

function drawFront(thFL,thFR){

ctF.clearRect(0,0,220,220);

grid(ctF,220,220);

var cx=110;
var gy=190;

var hx1=cx-HIP;
var hx2=cx+HIP;

var ay1=gy-Math.abs(Math.sin(thFL))*10;
var ay2=gy-Math.abs(Math.sin(thFR))*10;

var hy1=ay1-LEG;
var hy2=ay2-LEG;

// corpo preso às pernas
var bodyY=(hy1+hy2)/2;

ctF.fillStyle="#2c7be5";
ctF.fillRect(cx-BODYW/2,bodyY-BODYH,BODYW,BODYH);

// pernas
ctF.lineWidth=5;

ctF.strokeStyle="#1D9E75";
ctF.beginPath();
ctF.moveTo(hx1,hy1);
ctF.lineTo(hx1,ay1);
ctF.stroke();

ctF.strokeStyle="#378ADD";
ctF.beginPath();
ctF.moveTo(hx2,hy2);
ctF.lineTo(hx2,ay2);
ctF.stroke();

// pés inclinados
function foot(x,y,a,col){

ctF.save();

ctF.translate(x,y);
ctF.rotate(-a);

ctF.fillStyle=col;
ctF.fillRect(-18,-4,36,8);

ctF.restore();

}

foot(hx1,ay1,thFL,"#1D9E75");
foot(hx2,ay2,thFR,"#378ADD");

}

// =====================================================
// Trajetória global
// =====================================================

function drawGlobal(){

ctG.clearRect(0,0,220,220);

grid(ctG,220,220);

var cx=110;
var cy=110;

// trilha contínua
if(trail.length>1){

ctG.beginPath();

for(var i=0;i<trail.length;i++){

var p=trail[i];

var px=cx+p.x;
var py=cy-p.y;

if(i===0){
ctG.moveTo(px,py);
}else{
ctG.lineTo(px,py);
}

}

ctG.strokeStyle="#c0392b";
ctG.lineWidth=2;
ctG.stroke();

}

// robô atual
var px=cx+globalX;
var py=cy-globalY;

ctG.save();

ctG.translate(px,py);
ctG.rotate(heading);

ctG.fillStyle="#2c7be5";

ctG.beginPath();
ctG.moveTo(0,-10);
ctG.lineTo(7,8);
ctG.lineTo(-7,8);
ctG.closePath();
ctG.fill();

ctG.restore();

}

// =====================================================
// Loop
// =====================================================

function loop(ts){

requestAnimationFrame(loop);

var P=pars();

if(playing){

if(lastTS===null){
lastTS=ts;
}

var dt=ts-lastTS;

lastTS=ts;

simT+=dt;

}

var w=2*Math.PI*simT/P.T;

var thL=P.A*Math.sin(w);
var thR=P.A*Math.sin(w+Math.PI);

var thFL=P.F*Math.sin(w-Math.PI/2);
var thFR=P.F*Math.sin(w+Math.PI/2);

// =====================================================
// deslocamento
// =====================================================

if(playing){

var v=0.45;

if(mode==="wf"){

globalX+=v*Math.sin(heading);
globalY+=v*Math.cos(heading);

}

if(mode==="wb"){

globalX-=v*Math.sin(heading);
globalY-=v*Math.cos(heading);

}

// TURN corrigido
if(mode==="tl"){

heading-=0.018;

globalX+=v*Math.sin(heading);
globalY+=v*Math.cos(heading);

}

if(mode==="tr"){

heading+=0.018;

globalX+=v*Math.sin(heading);
globalY+=v*Math.cos(heading);

}

// trilha contínua
trail.push({
x:globalX,
y:globalY
});

if(trail.length>500){
trail.shift();
}

}

// =====================================================
// draw
// =====================================================

drawTop(thL,thR);
drawFront(thFL,thFR);
drawGlobal();

}

requestAnimationFrame(loop);

})();
</script>

""")